05

In [ ]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# ==========================================
# 1. ฟังก์ชันโหลดไฟล์จาก GitHub
# ==========================================
def load_csvs_from_github(owner, repo, path, ref="main", token=None):
    session = requests.Session()
    headers = {}
    if token:
        headers["Authorization"] = f"token {token}"
        
    api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}?ref={ref}"
    print(f"กำลังเชื่อมต่อ GitHub: {api_url}")
    
    resp = session.get(api_url, headers=headers)
    resp.raise_for_status()
    items = resp.json()
    
    # สนใจเฉพาะไฟล์ 101-104
    target_files = ['101.csv', '102.csv', '103.csv', '104.csv']
    csv_items = [it for it in items if it.get("type") == "file" and it.get("name", "").lower() in target_files]
    
    dfs = {}
    for it in csv_items:
        url = it.get("download_url") or f"https://raw.githubusercontent.com/{owner}/{repo}/{ref}/{path}/{it['name']}"
        print(f"กำลังดาวน์โหลดและอ่านไฟล์: {it['name']} ...")
        try:
            # ใช้ skiprows=4 เพื่อข้าม Header บรรทัดบน
            df = pd.read_csv(url, skiprows=4)
            df.columns = df.columns.str.strip() # ตัดช่องว่างด้านหน้า/หลังในชื่อคอลัมน์ออก
            
            # ลบแถวสุดท้ายที่มีคำว่า 'END' ทิ้ง
            df = df[df['Frequency(Hz)'] != 'END']
            
            # บังคับแปลงเป็นตัวเลข
            df['Frequency(Hz)'] = pd.to_numeric(df['Frequency(Hz)'], errors='coerce')
            df.dropna(subset=['Frequency(Hz)'], inplace=True)
            
            # เก็บไฟล์ด้วย Key ที่เป็นตัวพิมพ์ใหญ่ ('101.CSV', '102.CSV', ...)
            dfs[it["name"].upper()] = df
        except Exception as e:
            print(f"❌ Failed to read {it['name']}: {e}")
            
    return dfs

# ==========================================
# 2. ดึงข้อมูล
# ==========================================
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN") 
owner = "NuchPunnawichP"
repo = "Senior_Project_CU"
path = "ThirdDataset/05"  # ใส่โฟลเดอร์ใน Github ที่มีไฟล์ข้อมูล 101-104 ครับ

# โหลดข้อมูลมาไว้ในตัวแปร dataframes
dataframes = load_csvs_from_github(owner, repo, path, ref="main", token=GITHUB_TOKEN)
print(f"\n✅ ดึงไฟล์สำเร็จ {len(dataframes)} ไฟล์: {list(dataframes.keys())}\n")

# ==========================================
# 3. สร้างโฟลเดอร์ชื่อ "50X" สำหรับเก็บรูปภาพ
# ==========================================
FOLDER_NAME = "50X"
os.makedirs(FOLDER_NAME, exist_ok=True)
print(f"📁 สร้าง/ตรวจสอบโฟลเดอร์สำหรับบันทึกรูป: '{FOLDER_NAME}' เรียบร้อยแล้ว\n")

# ==========================================
# 4. วาดกราฟและบันทึกลงในโฟลเดอร์ทีละความถี่
# ==========================================
target_files = ['101.CSV', '102.CSV', '103.CSV', '104.CSV']
file_labels = ['101', '102', '103', '104']

# แกน X ใช้ตัวเลข 1, 2, 3, 4 (มี 4 ไฟล์) เพื่อคำนวณ Linear Regression ได้
x = np.arange(1, 5) 

# ดึงรายการความถี่ทั้งหมดจากไฟล์แรก (101.CSV) มาใช้เป็นหลัก 
# จะได้เป๊ะกับจุดทศนิยมที่มีอยู่ในไฟล์ ไม่ต้องกะเอง
if '101.CSV' in dataframes:
    frequencies = dataframes['101.CSV']['Frequency(Hz)'].dropna().unique()
else:
    frequencies = []

# วนลูปวาดกราฟตามความถี่ทั้งหมดที่มี
for freq in frequencies:
    cp_vals = []
    
    # ดึงค่า Cp จากไฟล์ 101 ถึง 104 ที่ความถี่เดียวกัน
    for f_name in target_files:
        if f_name in dataframes:
            df = dataframes[f_name]
            # ใช้ np.isclose เผื่อค่าทศนิยมคลาดเคลื่อน
            row = df[np.isclose(df['Frequency(Hz)'], freq)]
            if not row.empty:
                # เปลี่ยนไปดึงค่า 'Cp(F)-data' แทน
                cp_vals.append(row['Cp(F)-data'].values[0])
            else:
                cp_vals.append(np.nan)
        else:
            cp_vals.append(np.nan)
            
    y = np.array(cp_vals)
    valid_idx = ~np.isnan(y) # เช็คว่าจุดไหนไม่มีค่า (NaN)
    
    # ถ้าไม่มีข้อมูลในย่านความถี่นี้เลย ให้ข้ามไป จะได้ไม่ Error
    if np.sum(valid_idx) == 0:
        continue
    
    # วาดกราฟ
    plt.figure(figsize=(10, 6))
    plt.plot(x, y, 'bo-', label='Cp(F)-data', markersize=8)
    
    # หาค่าเทรนด์ (Linear Regression)
    if np.sum(valid_idx) > 1:
        slope, intercept, r_value, _, _ = linregress(x[valid_idx], y[valid_idx])
        plt.plot(x, slope * x + intercept, 'r--', label=f'Linear fit (R²={r_value**2:.4f})')
        title_text = f'Capacitance (Cp) Trend across files at {freq:.2f} Hz\nEquation: y = {slope:.2e}x + {intercept:.2e}'
    else:
        title_text = f'Capacitance (Cp) Trend across files at {freq:.2f} Hz'
        
    # ใส่ตัวเลขค่า Cp กํากับบนจุดแต่ละจุด
    for i, val in enumerate(y):
        if not np.isnan(val):
            plt.annotate(f'{val:.2e}', 
                         (x[i], val),
                         textcoords="offset points", 
                         xytext=(0, 15), 
                         ha='center', fontsize=10)

    # จัดการ Label 
    plt.title(title_text, fontsize=14)
    plt.xticks(x, file_labels, fontsize=12) # เปลี่ยนเลขแกน X ให้เป็นชื่อไฟล์ '101' ถึง '104'
    plt.xlabel('File Number (101 to 104)', fontsize=12)
    plt.ylabel('Cp(F)-data', fontsize=12) # เปลี่ยนชื่อแกน Y
    
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.tight_layout()
    
    # กำหนดชื่อไฟล์และนำไปต่อกับ Path โฟลเดอร์ (ใช้ชื่อไฟล์เป็นค่า int ของ freq)
    filename = os.path.join(FOLDER_NAME, f'trend_at_{int(freq)}Hz.png')
    
    # สั่งบันทึกไฟล์กราฟลงในโฟลเดอร์
    plt.savefig(filename)
    # print(f"✅ บันทึกกราฟ {filename}") # ขอซ่อนไว้เนื่องจากมันจะพิมพ์ออกมาทั้งหมด 100 บรรทัด
    
    # เคลียร์ Memory ออก ไม่ให้คอมค้าง
    plt.close()

print(f"\n🎉 บันทึกกราฟครบทั้งหมดเรียบร้อยแล้ว! (มีรูปทั้งหมด {len(frequencies)} รูปในโฟลเดอร์ '{FOLDER_NAME}')")

กำลังเชื่อมต่อ GitHub: https://api.github.com/repos/NuchPunnawichP/Senior_Project_CU/contents/ThirdDataset/05?ref=main
กำลังดาวน์โหลดและอ่านไฟล์: 101.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 102.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 103.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 104.CSV ...

✅ ดึงไฟล์สำเร็จ 4 ไฟล์: ['101.CSV', '102.CSV', '103.CSV', '104.CSV']

📁 สร้าง/ตรวจสอบโฟลเดอร์สำหรับบันทึกรูป: '10X' เรียบร้อยแล้ว


🎉 บันทึกกราฟครบทั้งหมดเรียบร้อยแล้ว! (มีรูปทั้งหมด 100 รูปในโฟลเดอร์ '10X')


In [2]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# ==========================================
# 1. ฟังก์ชันโหลดไฟล์จาก GitHub
# ==========================================
def load_csvs_from_github(owner, repo, path, ref="main", token=None):
    session = requests.Session()
    headers = {}
    if token:
        headers["Authorization"] = f"token {token}"
        
    api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}?ref={ref}"
    print(f"กำลังเชื่อมต่อ GitHub: {api_url}")
    
    resp = session.get(api_url, headers=headers)
    resp.raise_for_status()
    items = resp.json()
    
    # สนใจเฉพาะไฟล์ 201-204
    target_files = ['201.csv', '202.csv', '203.csv', '204.csv']
    csv_items = [it for it in items if it.get("type") == "file" and it.get("name", "").lower() in target_files]
    
    dfs = {}
    for it in csv_items:
        url = it.get("download_url") or f"https://raw.githubusercontent.com/{owner}/{repo}/{ref}/{path}/{it['name']}"
        print(f"กำลังดาวน์โหลดและอ่านไฟล์: {it['name']} ...")
        try:
            # ใช้ skiprows=4 เพื่อข้าม Header บรรทัดบน
            df = pd.read_csv(url, skiprows=4)
            df.columns = df.columns.str.strip() # ตัดช่องว่างด้านหน้า/หลังในชื่อคอลัมน์ออก
            
            # ลบแถวสุดท้ายที่มีคำว่า 'END' ทิ้ง
            df = df[df['Frequency(Hz)'] != 'END']
            
            # บังคับแปลงเป็นตัวเลข
            df['Frequency(Hz)'] = pd.to_numeric(df['Frequency(Hz)'], errors='coerce')
            df.dropna(subset=['Frequency(Hz)'], inplace=True)
            
            # เก็บไฟล์ด้วย Key ที่เป็นตัวพิมพ์ใหญ่ ('101.CSV', '102.CSV', ...)
            dfs[it["name"].upper()] = df
        except Exception as e:
            print(f"❌ Failed to read {it['name']}: {e}")
            
    return dfs

# ==========================================
# 2. ดึงข้อมูล
# ==========================================
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN") 
owner = "NuchPunnawichP"
repo = "Senior_Project_CU"
path = "ThirdDataset/05"  # ใส่โฟลเดอร์ใน Github ที่มีไฟล์ข้อมูล 101-104 ครับ

# โหลดข้อมูลมาไว้ในตัวแปร dataframes
dataframes = load_csvs_from_github(owner, repo, path, ref="main", token=GITHUB_TOKEN)
print(f"\n✅ ดึงไฟล์สำเร็จ {len(dataframes)} ไฟล์: {list(dataframes.keys())}\n")

# ==========================================
# 3. สร้างโฟลเดอร์ชื่อ "20X" สำหรับเก็บรูปภาพ
# ==========================================
FOLDER_NAME = "20X"
os.makedirs(FOLDER_NAME, exist_ok=True)
print(f"📁 สร้าง/ตรวจสอบโฟลเดอร์สำหรับบันทึกรูป: '{FOLDER_NAME}' เรียบร้อยแล้ว\n")

# ==========================================
# 4. วาดกราฟและบันทึกลงในโฟลเดอร์ทีละความถี่
# ==========================================
target_files = ['201.CSV', '202.CSV', '203.CSV', '204.CSV']
file_labels = ['201', '202', '203', '204']

# แกน X ใช้ตัวเลข 1, 2, 3, 4 (มี 4 ไฟล์) เพื่อคำนวณ Linear Regression ได้
x = np.arange(1, 5) 

# ดึงรายการความถี่ทั้งหมดจากไฟล์แรก (201.CSV) มาใช้เป็นหลัก 
# จะได้เป๊ะกับจุดทศนิยมที่มีอยู่ในไฟล์ ไม่ต้องกะเอง
if '201.CSV' in dataframes:
    frequencies = dataframes['201.CSV']['Frequency(Hz)'].dropna().unique()
else:
    frequencies = []

# วนลูปวาดกราฟตามความถี่ทั้งหมดที่มี
for freq in frequencies:
    cp_vals = []
    
    # ดึงค่า Cp จากไฟล์ 201 ถึง 204 ที่ความถี่เดียวกัน
    for f_name in target_files:
        if f_name in dataframes:
            df = dataframes[f_name]
            # ใช้ np.isclose เผื่อค่าทศนิยมคลาดเคลื่อน
            row = df[np.isclose(df['Frequency(Hz)'], freq)]
            if not row.empty:
                # เปลี่ยนไปดึงค่า 'Cp(F)-data' แทน
                cp_vals.append(row['Cp(F)-data'].values[0])
            else:
                cp_vals.append(np.nan)
        else:
            cp_vals.append(np.nan)
            
    y = np.array(cp_vals)
    valid_idx = ~np.isnan(y) # เช็คว่าจุดไหนไม่มีค่า (NaN)
    
    # ถ้าไม่มีข้อมูลในย่านความถี่นี้เลย ให้ข้ามไป จะได้ไม่ Error
    if np.sum(valid_idx) == 0:
        continue
    
    # วาดกราฟ
    plt.figure(figsize=(10, 6))
    plt.plot(x, y, 'bo-', label='Cp(F)-data', markersize=8)
    
    # หาค่าเทรนด์ (Linear Regression)
    if np.sum(valid_idx) > 1:
        slope, intercept, r_value, _, _ = linregress(x[valid_idx], y[valid_idx])
        plt.plot(x, slope * x + intercept, 'r--', label=f'Linear fit (R²={r_value**2:.4f})')
        title_text = f'Capacitance (Cp) Trend across files at {freq:.2f} Hz\nEquation: y = {slope:.2e}x + {intercept:.2e}'
    else:
        title_text = f'Capacitance (Cp) Trend across files at {freq:.2f} Hz'
        
    # ใส่ตัวเลขค่า Cp กํากับบนจุดแต่ละจุด
    for i, val in enumerate(y):
        if not np.isnan(val):
            plt.annotate(f'{val:.2e}', 
                         (x[i], val),
                         textcoords="offset points", 
                         xytext=(0, 15), 
                         ha='center', fontsize=10)

    # จัดการ Label 
    plt.title(title_text, fontsize=14)
    plt.xticks(x, file_labels, fontsize=12) # เปลี่ยนเลขแกน X ให้เป็นชื่อไฟล์ '101' ถึง '104'
    plt.xlabel('File Number (101 to 104)', fontsize=12)
    plt.ylabel('Cp(F)-data', fontsize=12) # เปลี่ยนชื่อแกน Y
    
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.tight_layout()
    
    # กำหนดชื่อไฟล์และนำไปต่อกับ Path โฟลเดอร์ (ใช้ชื่อไฟล์เป็นค่า int ของ freq)
    filename = os.path.join(FOLDER_NAME, f'trend_at_{int(freq)}Hz.png')
    
    # สั่งบันทึกไฟล์กราฟลงในโฟลเดอร์
    plt.savefig(filename)
    # print(f"✅ บันทึกกราฟ {filename}") # ขอซ่อนไว้เนื่องจากมันจะพิมพ์ออกมาทั้งหมด 100 บรรทัด
    
    # เคลียร์ Memory ออก ไม่ให้คอมค้าง
    plt.close()

print(f"\n🎉 บันทึกกราฟครบทั้งหมดเรียบร้อยแล้ว! (มีรูปทั้งหมด {len(frequencies)} รูปในโฟลเดอร์ '{FOLDER_NAME}')")

กำลังเชื่อมต่อ GitHub: https://api.github.com/repos/NuchPunnawichP/Senior_Project_CU/contents/ThirdDataset/05?ref=main
กำลังดาวน์โหลดและอ่านไฟล์: 201.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 202.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 203.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 204.CSV ...

✅ ดึงไฟล์สำเร็จ 4 ไฟล์: ['201.CSV', '202.CSV', '203.CSV', '204.CSV']

📁 สร้าง/ตรวจสอบโฟลเดอร์สำหรับบันทึกรูป: '20X' เรียบร้อยแล้ว


🎉 บันทึกกราฟครบทั้งหมดเรียบร้อยแล้ว! (มีรูปทั้งหมด 100 รูปในโฟลเดอร์ '20X')


In [3]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# ==========================================
# 1. ฟังก์ชันโหลดไฟล์จาก GitHub
# ==========================================
def load_csvs_from_github(owner, repo, path, ref="main", token=None):
    session = requests.Session()
    headers = {}
    if token:
        headers["Authorization"] = f"token {token}"
        
    api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}?ref={ref}"
    print(f"กำลังเชื่อมต่อ GitHub: {api_url}")
    
    resp = session.get(api_url, headers=headers)
    resp.raise_for_status()
    items = resp.json()
    
    # สนใจเฉพาะไฟล์ 301-304
    target_files = ['301.csv', '302.csv', '303.csv', '304.csv']
    csv_items = [it for it in items if it.get("type") == "file" and it.get("name", "").lower() in target_files]
    
    dfs = {}
    for it in csv_items:
        url = it.get("download_url") or f"https://raw.githubusercontent.com/{owner}/{repo}/{ref}/{path}/{it['name']}"
        print(f"กำลังดาวน์โหลดและอ่านไฟล์: {it['name']} ...")
        try:
            # ใช้ skiprows=4 เพื่อข้าม Header บรรทัดบน
            df = pd.read_csv(url, skiprows=4)
            df.columns = df.columns.str.strip() # ตัดช่องว่างด้านหน้า/หลังในชื่อคอลัมน์ออก
            
            # ลบแถวสุดท้ายที่มีคำว่า 'END' ทิ้ง
            df = df[df['Frequency(Hz)'] != 'END']
            
            # บังคับแปลงเป็นตัวเลข
            df['Frequency(Hz)'] = pd.to_numeric(df['Frequency(Hz)'], errors='coerce')
            df.dropna(subset=['Frequency(Hz)'], inplace=True)
            
            # เก็บไฟล์ด้วย Key ที่เป็นตัวพิมพ์ใหญ่ ('101.CSV', '102.CSV', ...)
            dfs[it["name"].upper()] = df
        except Exception as e:
            print(f"❌ Failed to read {it['name']}: {e}")
            
    return dfs

# ==========================================
# 2. ดึงข้อมูล
# ==========================================
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN") 
owner = "NuchPunnawichP"
repo = "Senior_Project_CU"
path = "ThirdDataset/05"  # ใส่โฟลเดอร์ใน Github ที่มีไฟล์ข้อมูล 101-104 ครับ

# โหลดข้อมูลมาไว้ในตัวแปร dataframes
dataframes = load_csvs_from_github(owner, repo, path, ref="main", token=GITHUB_TOKEN)
print(f"\n✅ ดึงไฟล์สำเร็จ {len(dataframes)} ไฟล์: {list(dataframes.keys())}\n")

# ==========================================
# 3. สร้างโฟลเดอร์ชื่อ "30X" สำหรับเก็บรูปภาพ
# ==========================================
FOLDER_NAME = "30X"
os.makedirs(FOLDER_NAME, exist_ok=True)
print(f"📁 สร้าง/ตรวจสอบโฟลเดอร์สำหรับบันทึกรูป: '{FOLDER_NAME}' เรียบร้อยแล้ว\n")

# ==========================================
# 4. วาดกราฟและบันทึกลงในโฟลเดอร์ทีละความถี่
# ==========================================
target_files = ['301.CSV', '302.CSV', '303.CSV', '304.CSV']
file_labels = ['301', '302', '303', '304']

# แกน X ใช้ตัวเลข 1, 2, 3, 4 (มี 4 ไฟล์) เพื่อคำนวณ Linear Regression ได้
x = np.arange(1, 5) 

# ดึงรายการความถี่ทั้งหมดจากไฟล์แรก (301.CSV) มาใช้เป็นหลัก 
# จะได้เป๊ะกับจุดทศนิยมที่มีอยู่ในไฟล์ ไม่ต้องกะเอง
if '301.CSV' in dataframes:
    frequencies = dataframes['301.CSV']['Frequency(Hz)'].dropna().unique()
else:
    frequencies = []

# วนลูปวาดกราฟตามความถี่ทั้งหมดที่มี
for freq in frequencies:
    cp_vals = []
    
    # ดึงค่า Cp จากไฟล์ 301 ถึง 304 ที่ความถี่เดียวกัน
    for f_name in target_files:
        if f_name in dataframes:
            df = dataframes[f_name]
            # ใช้ np.isclose เผื่อค่าทศนิยมคลาดเคลื่อน
            row = df[np.isclose(df['Frequency(Hz)'], freq)]
            if not row.empty:
                # เปลี่ยนไปดึงค่า 'Cp(F)-data' แทน
                cp_vals.append(row['Cp(F)-data'].values[0])
            else:
                cp_vals.append(np.nan)
        else:
            cp_vals.append(np.nan)
            
    y = np.array(cp_vals)
    valid_idx = ~np.isnan(y) # เช็คว่าจุดไหนไม่มีค่า (NaN)
    
    # ถ้าไม่มีข้อมูลในย่านความถี่นี้เลย ให้ข้ามไป จะได้ไม่ Error
    if np.sum(valid_idx) == 0:
        continue
    
    # วาดกราฟ
    plt.figure(figsize=(10, 6))
    plt.plot(x, y, 'bo-', label='Cp(F)-data', markersize=8)
    
    # หาค่าเทรนด์ (Linear Regression)
    if np.sum(valid_idx) > 1:
        slope, intercept, r_value, _, _ = linregress(x[valid_idx], y[valid_idx])
        plt.plot(x, slope * x + intercept, 'r--', label=f'Linear fit (R²={r_value**2:.4f})')
        title_text = f'Capacitance (Cp) Trend across files at {freq:.2f} Hz\nEquation: y = {slope:.2e}x + {intercept:.2e}'
    else:
        title_text = f'Capacitance (Cp) Trend across files at {freq:.2f} Hz'
        
    # ใส่ตัวเลขค่า Cp กํากับบนจุดแต่ละจุด
    for i, val in enumerate(y):
        if not np.isnan(val):
            plt.annotate(f'{val:.2e}', 
                         (x[i], val),
                         textcoords="offset points", 
                         xytext=(0, 15), 
                         ha='center', fontsize=10)

    # จัดการ Label 
    plt.title(title_text, fontsize=14)
    plt.xticks(x, file_labels, fontsize=12) # เปลี่ยนเลขแกน X ให้เป็นชื่อไฟล์ '101' ถึง '104'
    plt.xlabel('File Number (101 to 104)', fontsize=12)
    plt.ylabel('Cp(F)-data', fontsize=12) # เปลี่ยนชื่อแกน Y
    
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.tight_layout()
    
    # กำหนดชื่อไฟล์และนำไปต่อกับ Path โฟลเดอร์ (ใช้ชื่อไฟล์เป็นค่า int ของ freq)
    filename = os.path.join(FOLDER_NAME, f'trend_at_{int(freq)}Hz.png')
    
    # สั่งบันทึกไฟล์กราฟลงในโฟลเดอร์
    plt.savefig(filename)
    # print(f"✅ บันทึกกราฟ {filename}") # ขอซ่อนไว้เนื่องจากมันจะพิมพ์ออกมาทั้งหมด 100 บรรทัด
    
    # เคลียร์ Memory ออก ไม่ให้คอมค้าง
    plt.close()

print(f"\n🎉 บันทึกกราฟครบทั้งหมดเรียบร้อยแล้ว! (มีรูปทั้งหมด {len(frequencies)} รูปในโฟลเดอร์ '{FOLDER_NAME}')")

กำลังเชื่อมต่อ GitHub: https://api.github.com/repos/NuchPunnawichP/Senior_Project_CU/contents/ThirdDataset/05?ref=main
กำลังดาวน์โหลดและอ่านไฟล์: 301.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 302.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 303.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 304.CSV ...

✅ ดึงไฟล์สำเร็จ 4 ไฟล์: ['301.CSV', '302.CSV', '303.CSV', '304.CSV']

📁 สร้าง/ตรวจสอบโฟลเดอร์สำหรับบันทึกรูป: '30X' เรียบร้อยแล้ว


🎉 บันทึกกราฟครบทั้งหมดเรียบร้อยแล้ว! (มีรูปทั้งหมด 100 รูปในโฟลเดอร์ '30X')


In [4]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# ==========================================
# 1. ฟังก์ชันโหลดไฟล์จาก GitHub
# ==========================================
def load_csvs_from_github(owner, repo, path, ref="main", token=None):
    session = requests.Session()
    headers = {}
    if token:
        headers["Authorization"] = f"token {token}"
        
    api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}?ref={ref}"
    print(f"กำลังเชื่อมต่อ GitHub: {api_url}")
    
    resp = session.get(api_url, headers=headers)
    resp.raise_for_status()
    items = resp.json()
    
    # สนใจเฉพาะไฟล์ 401-404
    target_files = ['401.csv', '402.csv', '403.csv', '404.csv']
    csv_items = [it for it in items if it.get("type") == "file" and it.get("name", "").lower() in target_files]
    
    dfs = {}
    for it in csv_items:
        url = it.get("download_url") or f"https://raw.githubusercontent.com/{owner}/{repo}/{ref}/{path}/{it['name']}"
        print(f"กำลังดาวน์โหลดและอ่านไฟล์: {it['name']} ...")
        try:
            # ใช้ skiprows=4 เพื่อข้าม Header บรรทัดบน
            df = pd.read_csv(url, skiprows=4)
            df.columns = df.columns.str.strip() # ตัดช่องว่างด้านหน้า/หลังในชื่อคอลัมน์ออก
            
            # ลบแถวสุดท้ายที่มีคำว่า 'END' ทิ้ง
            df = df[df['Frequency(Hz)'] != 'END']
            
            # บังคับแปลงเป็นตัวเลข
            df['Frequency(Hz)'] = pd.to_numeric(df['Frequency(Hz)'], errors='coerce')
            df.dropna(subset=['Frequency(Hz)'], inplace=True)
            
            # เก็บไฟล์ด้วย Key ที่เป็นตัวพิมพ์ใหญ่ ('101.CSV', '102.CSV', ...)
            dfs[it["name"].upper()] = df
        except Exception as e:
            print(f"❌ Failed to read {it['name']}: {e}")
            
    return dfs

# ==========================================
# 2. ดึงข้อมูล
# ==========================================
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN") 
owner = "NuchPunnawichP"
repo = "Senior_Project_CU"
path = "ThirdDataset/05"  # ใส่โฟลเดอร์ใน Github ที่มีไฟล์ข้อมูล 101-104 ครับ

# โหลดข้อมูลมาไว้ในตัวแปร dataframes
dataframes = load_csvs_from_github(owner, repo, path, ref="main", token=GITHUB_TOKEN)
print(f"\n✅ ดึงไฟล์สำเร็จ {len(dataframes)} ไฟล์: {list(dataframes.keys())}\n")

# ==========================================
# 3. สร้างโฟลเดอร์ชื่อ "40X" สำหรับเก็บรูปภาพ
# ==========================================
FOLDER_NAME = "40X"
os.makedirs(FOLDER_NAME, exist_ok=True)
print(f"📁 สร้าง/ตรวจสอบโฟลเดอร์สำหรับบันทึกรูป: '{FOLDER_NAME}' เรียบร้อยแล้ว\n")

# ==========================================
# 4. วาดกราฟและบันทึกลงในโฟลเดอร์ทีละความถี่
# ==========================================
target_files = ['401.CSV', '402.CSV', '403.CSV', '404.CSV']
file_labels = ['401', '402', '403', '404']

# แกน X ใช้ตัวเลข 1, 2, 3, 4 (มี 4 ไฟล์) เพื่อคำนวณ Linear Regression ได้
x = np.arange(1, 5) 

# ดึงรายการความถี่ทั้งหมดจากไฟล์แรก (401.CSV) มาใช้เป็นหลัก 
# จะได้เป๊ะกับจุดทศนิยมที่มีอยู่ในไฟล์ ไม่ต้องกะเอง
if '401.CSV' in dataframes:
    frequencies = dataframes['401.CSV']['Frequency(Hz)'].dropna().unique()
else:
    frequencies = []

# วนลูปวาดกราฟตามความถี่ทั้งหมดที่มี
for freq in frequencies:
    cp_vals = []
    
    # ดึงค่า Cp จากไฟล์ 401 ถึง 404 ที่ความถี่เดียวกัน
    for f_name in target_files:
        if f_name in dataframes:
            df = dataframes[f_name]
            # ใช้ np.isclose เผื่อค่าทศนิยมคลาดเคลื่อน
            row = df[np.isclose(df['Frequency(Hz)'], freq)]
            if not row.empty:
                # เปลี่ยนไปดึงค่า 'Cp(F)-data' แทน
                cp_vals.append(row['Cp(F)-data'].values[0])
            else:
                cp_vals.append(np.nan)
        else:
            cp_vals.append(np.nan)
            
    y = np.array(cp_vals)
    valid_idx = ~np.isnan(y) # เช็คว่าจุดไหนไม่มีค่า (NaN)
    
    # ถ้าไม่มีข้อมูลในย่านความถี่นี้เลย ให้ข้ามไป จะได้ไม่ Error
    if np.sum(valid_idx) == 0:
        continue
    
    # วาดกราฟ
    plt.figure(figsize=(10, 6))
    plt.plot(x, y, 'bo-', label='Cp(F)-data', markersize=8)
    
    # หาค่าเทรนด์ (Linear Regression)
    if np.sum(valid_idx) > 1:
        slope, intercept, r_value, _, _ = linregress(x[valid_idx], y[valid_idx])
        plt.plot(x, slope * x + intercept, 'r--', label=f'Linear fit (R²={r_value**2:.4f})')
        title_text = f'Capacitance (Cp) Trend across files at {freq:.2f} Hz\nEquation: y = {slope:.2e}x + {intercept:.2e}'
    else:
        title_text = f'Capacitance (Cp) Trend across files at {freq:.2f} Hz'
        
    # ใส่ตัวเลขค่า Cp กํากับบนจุดแต่ละจุด
    for i, val in enumerate(y):
        if not np.isnan(val):
            plt.annotate(f'{val:.2e}', 
                         (x[i], val),
                         textcoords="offset points", 
                         xytext=(0, 15), 
                         ha='center', fontsize=10)

    # จัดการ Label 
    plt.title(title_text, fontsize=14)
    plt.xticks(x, file_labels, fontsize=12) # เปลี่ยนเลขแกน X ให้เป็นชื่อไฟล์ '101' ถึง '104'
    plt.xlabel('File Number (101 to 104)', fontsize=12)
    plt.ylabel('Cp(F)-data', fontsize=12) # เปลี่ยนชื่อแกน Y
    
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.tight_layout()
    
    # กำหนดชื่อไฟล์และนำไปต่อกับ Path โฟลเดอร์ (ใช้ชื่อไฟล์เป็นค่า int ของ freq)
    filename = os.path.join(FOLDER_NAME, f'trend_at_{int(freq)}Hz.png')
    
    # สั่งบันทึกไฟล์กราฟลงในโฟลเดอร์
    plt.savefig(filename)
    # print(f"✅ บันทึกกราฟ {filename}") # ขอซ่อนไว้เนื่องจากมันจะพิมพ์ออกมาทั้งหมด 100 บรรทัด
    
    # เคลียร์ Memory ออก ไม่ให้คอมค้าง
    plt.close()

print(f"\n🎉 บันทึกกราฟครบทั้งหมดเรียบร้อยแล้ว! (มีรูปทั้งหมด {len(frequencies)} รูปในโฟลเดอร์ '{FOLDER_NAME}')")

กำลังเชื่อมต่อ GitHub: https://api.github.com/repos/NuchPunnawichP/Senior_Project_CU/contents/ThirdDataset/05?ref=main
กำลังดาวน์โหลดและอ่านไฟล์: 401.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 402.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 403.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 404.CSV ...

✅ ดึงไฟล์สำเร็จ 4 ไฟล์: ['401.CSV', '402.CSV', '403.CSV', '404.CSV']

📁 สร้าง/ตรวจสอบโฟลเดอร์สำหรับบันทึกรูป: '40X' เรียบร้อยแล้ว


🎉 บันทึกกราฟครบทั้งหมดเรียบร้อยแล้ว! (มีรูปทั้งหมด 100 รูปในโฟลเดอร์ '40X')


06

In [7]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# ==========================================
# 1. ฟังก์ชันโหลดไฟล์จาก GitHub
# ==========================================
def load_csvs_from_github(owner, repo, path, ref="main", token=None):
    session = requests.Session()
    headers = {}
    if token:
        headers["Authorization"] = f"token {token}"
        
    api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}?ref={ref}"
    print(f"กำลังเชื่อมต่อ GitHub: {api_url}")
    
    resp = session.get(api_url, headers=headers)
    resp.raise_for_status()
    items = resp.json()
    
    # สนใจเฉพาะไฟล์ 101-104
    target_files = ['101.csv', '102.csv', '103.csv', '104.csv']
    csv_items = [it for it in items if it.get("type") == "file" and it.get("name", "").lower() in target_files]
    
    dfs = {}
    for it in csv_items:
        url = it.get("download_url") or f"https://raw.githubusercontent.com/{owner}/{repo}/{ref}/{path}/{it['name']}"
        print(f"กำลังดาวน์โหลดและอ่านไฟล์: {it['name']} ...")
        try:
            # ใช้ skiprows=4 เพื่อข้าม Header บรรทัดบน
            df = pd.read_csv(url, skiprows=4)
            df.columns = df.columns.str.strip() # ตัดช่องว่างด้านหน้า/หลังในชื่อคอลัมน์ออก
            
            # ลบแถวสุดท้ายที่มีคำว่า 'END' ทิ้ง
            df = df[df['Frequency(Hz)'] != 'END']
            
            # บังคับแปลงเป็นตัวเลข
            df['Frequency(Hz)'] = pd.to_numeric(df['Frequency(Hz)'], errors='coerce')
            df.dropna(subset=['Frequency(Hz)'], inplace=True)
            
            # เก็บไฟล์ด้วย Key ที่เป็นตัวพิมพ์ใหญ่ ('101.CSV', '102.CSV', ...)
            dfs[it["name"].upper()] = df
        except Exception as e:
            print(f"❌ Failed to read {it['name']}: {e}")
            
    return dfs

# ==========================================
# 2. ดึงข้อมูล
# ==========================================
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN") 
owner = "NuchPunnawichP"
repo = "Senior_Project_CU"
path = "ThirdDataset/06"  # ใส่โฟลเดอร์ใน Github ที่มีไฟล์ข้อมูล 101-104 ครับ

# โหลดข้อมูลมาไว้ในตัวแปร dataframes
dataframes = load_csvs_from_github(owner, repo, path, ref="main", token=GITHUB_TOKEN)
print(f"\n✅ ดึงไฟล์สำเร็จ {len(dataframes)} ไฟล์: {list(dataframes.keys())}\n")

# ==========================================
# 3. สร้างโฟลเดอร์ชื่อ "10X" สำหรับเก็บรูปภาพ
# ==========================================
FOLDER_NAME = "10X"
os.makedirs(FOLDER_NAME, exist_ok=True)
print(f"📁 สร้าง/ตรวจสอบโฟลเดอร์สำหรับบันทึกรูป: '{FOLDER_NAME}' เรียบร้อยแล้ว\n")

# ==========================================
# 4. วาดกราฟและบันทึกลงในโฟลเดอร์ทีละความถี่
# ==========================================
target_files = ['101.CSV', '102.CSV', '103.CSV', '104.CSV']
file_labels = ['101', '102', '103', '104']

# แกน X ใช้ตัวเลข 1, 2, 3, 4 (มี 4 ไฟล์) เพื่อคำนวณ Linear Regression ได้
x = np.arange(1, 5) 

# ดึงรายการความถี่ทั้งหมดจากไฟล์แรก (101.CSV) มาใช้เป็นหลัก 
# จะได้เป๊ะกับจุดทศนิยมที่มีอยู่ในไฟล์ ไม่ต้องกะเอง
if '101.CSV' in dataframes:
    frequencies = dataframes['101.CSV']['Frequency(Hz)'].dropna().unique()
else:
    frequencies = []

# วนลูปวาดกราฟตามความถี่ทั้งหมดที่มี
for freq in frequencies:
    cp_vals = []
    
    # ดึงค่า Cp จากไฟล์ 401 ถึง 404 ที่ความถี่เดียวกัน
    for f_name in target_files:
        if f_name in dataframes:
            df = dataframes[f_name]
            # ใช้ np.isclose เผื่อค่าทศนิยมคลาดเคลื่อน
            row = df[np.isclose(df['Frequency(Hz)'], freq)]
            if not row.empty:
                # เปลี่ยนไปดึงค่า 'Cp(F)-data' แทน
                cp_vals.append(row['Cp(F)-data'].values[0])
            else:
                cp_vals.append(np.nan)
        else:
            cp_vals.append(np.nan)
            
    y = np.array(cp_vals)
    valid_idx = ~np.isnan(y) # เช็คว่าจุดไหนไม่มีค่า (NaN)
    
    # ถ้าไม่มีข้อมูลในย่านความถี่นี้เลย ให้ข้ามไป จะได้ไม่ Error
    if np.sum(valid_idx) == 0:
        continue
    
    # วาดกราฟ
    plt.figure(figsize=(10, 6))
    plt.plot(x, y, 'bo-', label='Cp(F)-data', markersize=8)
    
    # หาค่าเทรนด์ (Linear Regression)
    if np.sum(valid_idx) > 1:
        slope, intercept, r_value, _, _ = linregress(x[valid_idx], y[valid_idx])
        plt.plot(x, slope * x + intercept, 'r--', label=f'Linear fit (R²={r_value**2:.4f})')
        title_text = f'Capacitance (Cp) Trend across files at {freq:.2f} Hz\nEquation: y = {slope:.2e}x + {intercept:.2e}'
    else:
        title_text = f'Capacitance (Cp) Trend across files at {freq:.2f} Hz'
        
    # ใส่ตัวเลขค่า Cp กํากับบนจุดแต่ละจุด
    for i, val in enumerate(y):
        if not np.isnan(val):
            plt.annotate(f'{val:.2e}', 
                         (x[i], val),
                         textcoords="offset points", 
                         xytext=(0, 15), 
                         ha='center', fontsize=10)

    # จัดการ Label 
    plt.title(title_text, fontsize=14)
    plt.xticks(x, file_labels, fontsize=12) # เปลี่ยนเลขแกน X ให้เป็นชื่อไฟล์ '101' ถึง '104'
    plt.xlabel('File Number (101 to 104)', fontsize=12)
    plt.ylabel('Cp(F)-data', fontsize=12) # เปลี่ยนชื่อแกน Y
    
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.tight_layout()
    
    # กำหนดชื่อไฟล์และนำไปต่อกับ Path โฟลเดอร์ (ใช้ชื่อไฟล์เป็นค่า int ของ freq)
    filename = os.path.join(FOLDER_NAME, f'trend_at_{int(freq)}Hz.png')
    
    # สั่งบันทึกไฟล์กราฟลงในโฟลเดอร์
    plt.savefig(filename)
    # print(f"✅ บันทึกกราฟ {filename}") # ขอซ่อนไว้เนื่องจากมันจะพิมพ์ออกมาทั้งหมด 100 บรรทัด
    
    # เคลียร์ Memory ออก ไม่ให้คอมค้าง
    plt.close()

print(f"\n🎉 บันทึกกราฟครบทั้งหมดเรียบร้อยแล้ว! (มีรูปทั้งหมด {len(frequencies)} รูปในโฟลเดอร์ '{FOLDER_NAME}')")

กำลังเชื่อมต่อ GitHub: https://api.github.com/repos/NuchPunnawichP/Senior_Project_CU/contents/ThirdDataset/06?ref=main
กำลังดาวน์โหลดและอ่านไฟล์: 101.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 102.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 103.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 104.CSV ...

✅ ดึงไฟล์สำเร็จ 4 ไฟล์: ['101.CSV', '102.CSV', '103.CSV', '104.CSV']

📁 สร้าง/ตรวจสอบโฟลเดอร์สำหรับบันทึกรูป: '10X' เรียบร้อยแล้ว


🎉 บันทึกกราฟครบทั้งหมดเรียบร้อยแล้ว! (มีรูปทั้งหมด 100 รูปในโฟลเดอร์ '10X')


In [8]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# ==========================================
# 1. ฟังก์ชันโหลดไฟล์จาก GitHub
# ==========================================
def load_csvs_from_github(owner, repo, path, ref="main", token=None):
    session = requests.Session()
    headers = {}
    if token:
        headers["Authorization"] = f"token {token}"
        
    api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}?ref={ref}"
    print(f"กำลังเชื่อมต่อ GitHub: {api_url}")
    
    resp = session.get(api_url, headers=headers)
    resp.raise_for_status()
    items = resp.json()
    
    # สนใจเฉพาะไฟล์ 201-204
    target_files = ['201.csv', '202.csv', '203.csv', '204.csv']
    csv_items = [it for it in items if it.get("type") == "file" and it.get("name", "").lower() in target_files]
    
    dfs = {}
    for it in csv_items:
        url = it.get("download_url") or f"https://raw.githubusercontent.com/{owner}/{repo}/{ref}/{path}/{it['name']}"
        print(f"กำลังดาวน์โหลดและอ่านไฟล์: {it['name']} ...")
        try:
            # ใช้ skiprows=4 เพื่อข้าม Header บรรทัดบน
            df = pd.read_csv(url, skiprows=4)
            df.columns = df.columns.str.strip() # ตัดช่องว่างด้านหน้า/หลังในชื่อคอลัมน์ออก
            
            # ลบแถวสุดท้ายที่มีคำว่า 'END' ทิ้ง
            df = df[df['Frequency(Hz)'] != 'END']
            
            # บังคับแปลงเป็นตัวเลข
            df['Frequency(Hz)'] = pd.to_numeric(df['Frequency(Hz)'], errors='coerce')
            df.dropna(subset=['Frequency(Hz)'], inplace=True)
            
            # เก็บไฟล์ด้วย Key ที่เป็นตัวพิมพ์ใหญ่ ('101.CSV', '102.CSV', ...)
            dfs[it["name"].upper()] = df
        except Exception as e:
            print(f"❌ Failed to read {it['name']}: {e}")
            
    return dfs

# ==========================================
# 2. ดึงข้อมูล
# ==========================================
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN") 
owner = "NuchPunnawichP"
repo = "Senior_Project_CU"
path = "ThirdDataset/06"  # ใส่โฟลเดอร์ใน Github ที่มีไฟล์ข้อมูล 101-104 ครับ

# โหลดข้อมูลมาไว้ในตัวแปร dataframes
dataframes = load_csvs_from_github(owner, repo, path, ref="main", token=GITHUB_TOKEN)
print(f"\n✅ ดึงไฟล์สำเร็จ {len(dataframes)} ไฟล์: {list(dataframes.keys())}\n")

# ==========================================
# 3. สร้างโฟลเดอร์ชื่อ "20X" สำหรับเก็บรูปภาพ
# ==========================================
FOLDER_NAME = "20X"
os.makedirs(FOLDER_NAME, exist_ok=True)
print(f"📁 สร้าง/ตรวจสอบโฟลเดอร์สำหรับบันทึกรูป: '{FOLDER_NAME}' เรียบร้อยแล้ว\n")

# ==========================================
# 4. วาดกราฟและบันทึกลงในโฟลเดอร์ทีละความถี่
# ==========================================
target_files = ['201.CSV', '202.CSV', '203.CSV', '204.CSV']
file_labels = ['201', '202', '203', '204']

# แกน X ใช้ตัวเลข 1, 2, 3, 4 (มี 4 ไฟล์) เพื่อคำนวณ Linear Regression ได้
x = np.arange(1, 5) 

# ดึงรายการความถี่ทั้งหมดจากไฟล์แรก (201.CSV) มาใช้เป็นหลัก 
# จะได้เป๊ะกับจุดทศนิยมที่มีอยู่ในไฟล์ ไม่ต้องกะเอง
if '201.CSV' in dataframes:
    frequencies = dataframes['201.CSV']['Frequency(Hz)'].dropna().unique()
else:
    frequencies = []

# วนลูปวาดกราฟตามความถี่ทั้งหมดที่มี
for freq in frequencies:
    cp_vals = []
    
    # ดึงค่า Cp จากไฟล์ 401 ถึง 404 ที่ความถี่เดียวกัน
    for f_name in target_files:
        if f_name in dataframes:
            df = dataframes[f_name]
            # ใช้ np.isclose เผื่อค่าทศนิยมคลาดเคลื่อน
            row = df[np.isclose(df['Frequency(Hz)'], freq)]
            if not row.empty:
                # เปลี่ยนไปดึงค่า 'Cp(F)-data' แทน
                cp_vals.append(row['Cp(F)-data'].values[0])
            else:
                cp_vals.append(np.nan)
        else:
            cp_vals.append(np.nan)
            
    y = np.array(cp_vals)
    valid_idx = ~np.isnan(y) # เช็คว่าจุดไหนไม่มีค่า (NaN)
    
    # ถ้าไม่มีข้อมูลในย่านความถี่นี้เลย ให้ข้ามไป จะได้ไม่ Error
    if np.sum(valid_idx) == 0:
        continue
    
    # วาดกราฟ
    plt.figure(figsize=(10, 6))
    plt.plot(x, y, 'bo-', label='Cp(F)-data', markersize=8)
    
    # หาค่าเทรนด์ (Linear Regression)
    if np.sum(valid_idx) > 1:
        slope, intercept, r_value, _, _ = linregress(x[valid_idx], y[valid_idx])
        plt.plot(x, slope * x + intercept, 'r--', label=f'Linear fit (R²={r_value**2:.4f})')
        title_text = f'Capacitance (Cp) Trend across files at {freq:.2f} Hz\nEquation: y = {slope:.2e}x + {intercept:.2e}'
    else:
        title_text = f'Capacitance (Cp) Trend across files at {freq:.2f} Hz'
        
    # ใส่ตัวเลขค่า Cp กํากับบนจุดแต่ละจุด
    for i, val in enumerate(y):
        if not np.isnan(val):
            plt.annotate(f'{val:.2e}', 
                         (x[i], val),
                         textcoords="offset points", 
                         xytext=(0, 15), 
                         ha='center', fontsize=10)

    # จัดการ Label 
    plt.title(title_text, fontsize=14)
    plt.xticks(x, file_labels, fontsize=12) # เปลี่ยนเลขแกน X ให้เป็นชื่อไฟล์ '101' ถึง '104'
    plt.xlabel('File Number (101 to 104)', fontsize=12)
    plt.ylabel('Cp(F)-data', fontsize=12) # เปลี่ยนชื่อแกน Y
    
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.tight_layout()
    
    # กำหนดชื่อไฟล์และนำไปต่อกับ Path โฟลเดอร์ (ใช้ชื่อไฟล์เป็นค่า int ของ freq)
    filename = os.path.join(FOLDER_NAME, f'trend_at_{int(freq)}Hz.png')
    
    # สั่งบันทึกไฟล์กราฟลงในโฟลเดอร์
    plt.savefig(filename)
    # print(f"✅ บันทึกกราฟ {filename}") # ขอซ่อนไว้เนื่องจากมันจะพิมพ์ออกมาทั้งหมด 100 บรรทัด
    
    # เคลียร์ Memory ออก ไม่ให้คอมค้าง
    plt.close()

print(f"\n🎉 บันทึกกราฟครบทั้งหมดเรียบร้อยแล้ว! (มีรูปทั้งหมด {len(frequencies)} รูปในโฟลเดอร์ '{FOLDER_NAME}')")

กำลังเชื่อมต่อ GitHub: https://api.github.com/repos/NuchPunnawichP/Senior_Project_CU/contents/ThirdDataset/06?ref=main
กำลังดาวน์โหลดและอ่านไฟล์: 201.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 202.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 203.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 204.CSV ...

✅ ดึงไฟล์สำเร็จ 4 ไฟล์: ['201.CSV', '202.CSV', '203.CSV', '204.CSV']

📁 สร้าง/ตรวจสอบโฟลเดอร์สำหรับบันทึกรูป: '20X' เรียบร้อยแล้ว


🎉 บันทึกกราฟครบทั้งหมดเรียบร้อยแล้ว! (มีรูปทั้งหมด 100 รูปในโฟลเดอร์ '20X')


In [9]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# ==========================================
# 1. ฟังก์ชันโหลดไฟล์จาก GitHub
# ==========================================
def load_csvs_from_github(owner, repo, path, ref="main", token=None):
    session = requests.Session()
    headers = {}
    if token:
        headers["Authorization"] = f"token {token}"
        
    api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}?ref={ref}"
    print(f"กำลังเชื่อมต่อ GitHub: {api_url}")
    
    resp = session.get(api_url, headers=headers)
    resp.raise_for_status()
    items = resp.json()
    
    # สนใจเฉพาะไฟล์ 301-304
    target_files = ['301.csv', '302.csv', '303.csv', '304.csv']
    csv_items = [it for it in items if it.get("type") == "file" and it.get("name", "").lower() in target_files]
    
    dfs = {}
    for it in csv_items:
        url = it.get("download_url") or f"https://raw.githubusercontent.com/{owner}/{repo}/{ref}/{path}/{it['name']}"
        print(f"กำลังดาวน์โหลดและอ่านไฟล์: {it['name']} ...")
        try:
            # ใช้ skiprows=4 เพื่อข้าม Header บรรทัดบน
            df = pd.read_csv(url, skiprows=4)
            df.columns = df.columns.str.strip() # ตัดช่องว่างด้านหน้า/หลังในชื่อคอลัมน์ออก
            
            # ลบแถวสุดท้ายที่มีคำว่า 'END' ทิ้ง
            df = df[df['Frequency(Hz)'] != 'END']
            
            # บังคับแปลงเป็นตัวเลข
            df['Frequency(Hz)'] = pd.to_numeric(df['Frequency(Hz)'], errors='coerce')
            df.dropna(subset=['Frequency(Hz)'], inplace=True)
            
            # เก็บไฟล์ด้วย Key ที่เป็นตัวพิมพ์ใหญ่ ('101.CSV', '102.CSV', ...)
            dfs[it["name"].upper()] = df
        except Exception as e:
            print(f"❌ Failed to read {it['name']}: {e}")
            
    return dfs

# ==========================================
# 2. ดึงข้อมูล
# ==========================================
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN") 
owner = "NuchPunnawichP"
repo = "Senior_Project_CU"
path = "ThirdDataset/06"  # ใส่โฟลเดอร์ใน Github ที่มีไฟล์ข้อมูล 101-104 ครับ

# โหลดข้อมูลมาไว้ในตัวแปร dataframes
dataframes = load_csvs_from_github(owner, repo, path, ref="main", token=GITHUB_TOKEN)
print(f"\n✅ ดึงไฟล์สำเร็จ {len(dataframes)} ไฟล์: {list(dataframes.keys())}\n")

# ==========================================
# 3. สร้างโฟลเดอร์ชื่อ "30X" สำหรับเก็บรูปภาพ
# ==========================================
FOLDER_NAME = "30X"
os.makedirs(FOLDER_NAME, exist_ok=True)
print(f"📁 สร้าง/ตรวจสอบโฟลเดอร์สำหรับบันทึกรูป: '{FOLDER_NAME}' เรียบร้อยแล้ว\n")

# ==========================================
# 4. วาดกราฟและบันทึกลงในโฟลเดอร์ทีละความถี่
# ==========================================
target_files = ['301.CSV', '302.CSV', '303.CSV', '304.CSV']
file_labels = ['301', '302', '303', '304']

# แกน X ใช้ตัวเลข 1, 2, 3, 4 (มี 4 ไฟล์) เพื่อคำนวณ Linear Regression ได้
x = np.arange(1, 5) 

# ดึงรายการความถี่ทั้งหมดจากไฟล์แรก (301.CSV) มาใช้เป็นหลัก 
# จะได้เป๊ะกับจุดทศนิยมที่มีอยู่ในไฟล์ ไม่ต้องกะเอง
if '301.CSV' in dataframes:
    frequencies = dataframes['301.CSV']['Frequency(Hz)'].dropna().unique()
else:
    frequencies = []

# วนลูปวาดกราฟตามความถี่ทั้งหมดที่มี
for freq in frequencies:
    cp_vals = []
    
    # ดึงค่า Cp จากไฟล์ 401 ถึง 404 ที่ความถี่เดียวกัน
    for f_name in target_files:
        if f_name in dataframes:
            df = dataframes[f_name]
            # ใช้ np.isclose เผื่อค่าทศนิยมคลาดเคลื่อน
            row = df[np.isclose(df['Frequency(Hz)'], freq)]
            if not row.empty:
                # เปลี่ยนไปดึงค่า 'Cp(F)-data' แทน
                cp_vals.append(row['Cp(F)-data'].values[0])
            else:
                cp_vals.append(np.nan)
        else:
            cp_vals.append(np.nan)
            
    y = np.array(cp_vals)
    valid_idx = ~np.isnan(y) # เช็คว่าจุดไหนไม่มีค่า (NaN)
    
    # ถ้าไม่มีข้อมูลในย่านความถี่นี้เลย ให้ข้ามไป จะได้ไม่ Error
    if np.sum(valid_idx) == 0:
        continue
    
    # วาดกราฟ
    plt.figure(figsize=(10, 6))
    plt.plot(x, y, 'bo-', label='Cp(F)-data', markersize=8)
    
    # หาค่าเทรนด์ (Linear Regression)
    if np.sum(valid_idx) > 1:
        slope, intercept, r_value, _, _ = linregress(x[valid_idx], y[valid_idx])
        plt.plot(x, slope * x + intercept, 'r--', label=f'Linear fit (R²={r_value**2:.4f})')
        title_text = f'Capacitance (Cp) Trend across files at {freq:.2f} Hz\nEquation: y = {slope:.2e}x + {intercept:.2e}'
    else:
        title_text = f'Capacitance (Cp) Trend across files at {freq:.2f} Hz'
        
    # ใส่ตัวเลขค่า Cp กํากับบนจุดแต่ละจุด
    for i, val in enumerate(y):
        if not np.isnan(val):
            plt.annotate(f'{val:.2e}', 
                         (x[i], val),
                         textcoords="offset points", 
                         xytext=(0, 15), 
                         ha='center', fontsize=10)

    # จัดการ Label 
    plt.title(title_text, fontsize=14)
    plt.xticks(x, file_labels, fontsize=12) # เปลี่ยนเลขแกน X ให้เป็นชื่อไฟล์ '101' ถึง '104'
    plt.xlabel('File Number (101 to 104)', fontsize=12)
    plt.ylabel('Cp(F)-data', fontsize=12) # เปลี่ยนชื่อแกน Y
    
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.tight_layout()
    
    # กำหนดชื่อไฟล์และนำไปต่อกับ Path โฟลเดอร์ (ใช้ชื่อไฟล์เป็นค่า int ของ freq)
    filename = os.path.join(FOLDER_NAME, f'trend_at_{int(freq)}Hz.png')
    
    # สั่งบันทึกไฟล์กราฟลงในโฟลเดอร์
    plt.savefig(filename)
    # print(f"✅ บันทึกกราฟ {filename}") # ขอซ่อนไว้เนื่องจากมันจะพิมพ์ออกมาทั้งหมด 100 บรรทัด
    
    # เคลียร์ Memory ออก ไม่ให้คอมค้าง
    plt.close()

print(f"\n🎉 บันทึกกราฟครบทั้งหมดเรียบร้อยแล้ว! (มีรูปทั้งหมด {len(frequencies)} รูปในโฟลเดอร์ '{FOLDER_NAME}')")

กำลังเชื่อมต่อ GitHub: https://api.github.com/repos/NuchPunnawichP/Senior_Project_CU/contents/ThirdDataset/06?ref=main
กำลังดาวน์โหลดและอ่านไฟล์: 301.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 302.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 303.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 304.CSV ...

✅ ดึงไฟล์สำเร็จ 4 ไฟล์: ['301.CSV', '302.CSV', '303.CSV', '304.CSV']

📁 สร้าง/ตรวจสอบโฟลเดอร์สำหรับบันทึกรูป: '30X' เรียบร้อยแล้ว


🎉 บันทึกกราฟครบทั้งหมดเรียบร้อยแล้ว! (มีรูปทั้งหมด 100 รูปในโฟลเดอร์ '30X')


In [10]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# ==========================================
# 1. ฟังก์ชันโหลดไฟล์จาก GitHub
# ==========================================
def load_csvs_from_github(owner, repo, path, ref="main", token=None):
    session = requests.Session()
    headers = {}
    if token:
        headers["Authorization"] = f"token {token}"
        
    api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}?ref={ref}"
    print(f"กำลังเชื่อมต่อ GitHub: {api_url}")
    
    resp = session.get(api_url, headers=headers)
    resp.raise_for_status()
    items = resp.json()
    
    # สนใจเฉพาะไฟล์ 401-404
    target_files = ['401.csv', '402.csv', '403.csv', '404.csv']
    csv_items = [it for it in items if it.get("type") == "file" and it.get("name", "").lower() in target_files]
    
    dfs = {}
    for it in csv_items:
        url = it.get("download_url") or f"https://raw.githubusercontent.com/{owner}/{repo}/{ref}/{path}/{it['name']}"
        print(f"กำลังดาวน์โหลดและอ่านไฟล์: {it['name']} ...")
        try:
            # ใช้ skiprows=4 เพื่อข้าม Header บรรทัดบน
            df = pd.read_csv(url, skiprows=4)
            df.columns = df.columns.str.strip() # ตัดช่องว่างด้านหน้า/หลังในชื่อคอลัมน์ออก
            
            # ลบแถวสุดท้ายที่มีคำว่า 'END' ทิ้ง
            df = df[df['Frequency(Hz)'] != 'END']
            
            # บังคับแปลงเป็นตัวเลข
            df['Frequency(Hz)'] = pd.to_numeric(df['Frequency(Hz)'], errors='coerce')
            df.dropna(subset=['Frequency(Hz)'], inplace=True)
            
            # เก็บไฟล์ด้วย Key ที่เป็นตัวพิมพ์ใหญ่ ('101.CSV', '102.CSV', ...)
            dfs[it["name"].upper()] = df
        except Exception as e:
            print(f"❌ Failed to read {it['name']}: {e}")
            
    return dfs

# ==========================================
# 2. ดึงข้อมูล
# ==========================================
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN") 
owner = "NuchPunnawichP"
repo = "Senior_Project_CU"
path = "ThirdDataset/06"  # ใส่โฟลเดอร์ใน Github ที่มีไฟล์ข้อมูล 101-104 ครับ

# โหลดข้อมูลมาไว้ในตัวแปร dataframes
dataframes = load_csvs_from_github(owner, repo, path, ref="main", token=GITHUB_TOKEN)
print(f"\n✅ ดึงไฟล์สำเร็จ {len(dataframes)} ไฟล์: {list(dataframes.keys())}\n")

# ==========================================
# 3. สร้างโฟลเดอร์ชื่อ "40X" สำหรับเก็บรูปภาพ
# ==========================================
FOLDER_NAME = "40X"
os.makedirs(FOLDER_NAME, exist_ok=True)
print(f"📁 สร้าง/ตรวจสอบโฟลเดอร์สำหรับบันทึกรูป: '{FOLDER_NAME}' เรียบร้อยแล้ว\n")

# ==========================================
# 4. วาดกราฟและบันทึกลงในโฟลเดอร์ทีละความถี่
# ==========================================
target_files = ['401.CSV', '402.CSV', '403.CSV', '404.CSV']
file_labels = ['401', '402', '403', '404']

# แกน X ใช้ตัวเลข 1, 2, 3, 4 (มี 4 ไฟล์) เพื่อคำนวณ Linear Regression ได้
x = np.arange(1, 5) 

# ดึงรายการความถี่ทั้งหมดจากไฟล์แรก (401.CSV) มาใช้เป็นหลัก 
# จะได้เป๊ะกับจุดทศนิยมที่มีอยู่ในไฟล์ ไม่ต้องกะเอง
if '401.CSV' in dataframes:
    frequencies = dataframes['401.CSV']['Frequency(Hz)'].dropna().unique()
else:
    frequencies = []

# วนลูปวาดกราฟตามความถี่ทั้งหมดที่มี
for freq in frequencies:
    cp_vals = []
    
    # ดึงค่า Cp จากไฟล์ 401 ถึง 404 ที่ความถี่เดียวกัน
    for f_name in target_files:
        if f_name in dataframes:
            df = dataframes[f_name]
            # ใช้ np.isclose เผื่อค่าทศนิยมคลาดเคลื่อน
            row = df[np.isclose(df['Frequency(Hz)'], freq)]
            if not row.empty:
                # เปลี่ยนไปดึงค่า 'Cp(F)-data' แทน
                cp_vals.append(row['Cp(F)-data'].values[0])
            else:
                cp_vals.append(np.nan)
        else:
            cp_vals.append(np.nan)
            
    y = np.array(cp_vals)
    valid_idx = ~np.isnan(y) # เช็คว่าจุดไหนไม่มีค่า (NaN)
    
    # ถ้าไม่มีข้อมูลในย่านความถี่นี้เลย ให้ข้ามไป จะได้ไม่ Error
    if np.sum(valid_idx) == 0:
        continue
    
    # วาดกราฟ
    plt.figure(figsize=(10, 6))
    plt.plot(x, y, 'bo-', label='Cp(F)-data', markersize=8)
    
    # หาค่าเทรนด์ (Linear Regression)
    if np.sum(valid_idx) > 1:
        slope, intercept, r_value, _, _ = linregress(x[valid_idx], y[valid_idx])
        plt.plot(x, slope * x + intercept, 'r--', label=f'Linear fit (R²={r_value**2:.4f})')
        title_text = f'Capacitance (Cp) Trend across files at {freq:.2f} Hz\nEquation: y = {slope:.2e}x + {intercept:.2e}'
    else:
        title_text = f'Capacitance (Cp) Trend across files at {freq:.2f} Hz'
        
    # ใส่ตัวเลขค่า Cp กํากับบนจุดแต่ละจุด
    for i, val in enumerate(y):
        if not np.isnan(val):
            plt.annotate(f'{val:.2e}', 
                         (x[i], val),
                         textcoords="offset points", 
                         xytext=(0, 15), 
                         ha='center', fontsize=10)

    # จัดการ Label 
    plt.title(title_text, fontsize=14)
    plt.xticks(x, file_labels, fontsize=12) # เปลี่ยนเลขแกน X ให้เป็นชื่อไฟล์ '101' ถึง '104'
    plt.xlabel('File Number (101 to 104)', fontsize=12)
    plt.ylabel('Cp(F)-data', fontsize=12) # เปลี่ยนชื่อแกน Y
    
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.tight_layout()
    
    # กำหนดชื่อไฟล์และนำไปต่อกับ Path โฟลเดอร์ (ใช้ชื่อไฟล์เป็นค่า int ของ freq)
    filename = os.path.join(FOLDER_NAME, f'trend_at_{int(freq)}Hz.png')
    
    # สั่งบันทึกไฟล์กราฟลงในโฟลเดอร์
    plt.savefig(filename)
    # print(f"✅ บันทึกกราฟ {filename}") # ขอซ่อนไว้เนื่องจากมันจะพิมพ์ออกมาทั้งหมด 100 บรรทัด
    
    # เคลียร์ Memory ออก ไม่ให้คอมค้าง
    plt.close()

print(f"\n🎉 บันทึกกราฟครบทั้งหมดเรียบร้อยแล้ว! (มีรูปทั้งหมด {len(frequencies)} รูปในโฟลเดอร์ '{FOLDER_NAME}')")

กำลังเชื่อมต่อ GitHub: https://api.github.com/repos/NuchPunnawichP/Senior_Project_CU/contents/ThirdDataset/06?ref=main
กำลังดาวน์โหลดและอ่านไฟล์: 401.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 402.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 403.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 404.CSV ...

✅ ดึงไฟล์สำเร็จ 4 ไฟล์: ['401.CSV', '402.CSV', '403.CSV', '404.CSV']

📁 สร้าง/ตรวจสอบโฟลเดอร์สำหรับบันทึกรูป: '40X' เรียบร้อยแล้ว


🎉 บันทึกกราฟครบทั้งหมดเรียบร้อยแล้ว! (มีรูปทั้งหมด 100 รูปในโฟลเดอร์ '40X')


07

In [11]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# ==========================================
# 1. ฟังก์ชันโหลดไฟล์จาก GitHub
# ==========================================
def load_csvs_from_github(owner, repo, path, ref="main", token=None):
    session = requests.Session()
    headers = {}
    if token:
        headers["Authorization"] = f"token {token}"
        
    api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}?ref={ref}"
    print(f"กำลังเชื่อมต่อ GitHub: {api_url}")
    
    resp = session.get(api_url, headers=headers)
    resp.raise_for_status()
    items = resp.json()
    
    # สนใจเฉพาะไฟล์ 101-104
    target_files = ['101.csv', '102.csv', '103.csv', '104.csv']
    csv_items = [it for it in items if it.get("type") == "file" and it.get("name", "").lower() in target_files]
    
    dfs = {}
    for it in csv_items:
        url = it.get("download_url") or f"https://raw.githubusercontent.com/{owner}/{repo}/{ref}/{path}/{it['name']}"
        print(f"กำลังดาวน์โหลดและอ่านไฟล์: {it['name']} ...")
        try:
            # ใช้ skiprows=4 เพื่อข้าม Header บรรทัดบน
            df = pd.read_csv(url, skiprows=4)
            df.columns = df.columns.str.strip() # ตัดช่องว่างด้านหน้า/หลังในชื่อคอลัมน์ออก
            
            # ลบแถวสุดท้ายที่มีคำว่า 'END' ทิ้ง
            df = df[df['Frequency(Hz)'] != 'END']
            
            # บังคับแปลงเป็นตัวเลข
            df['Frequency(Hz)'] = pd.to_numeric(df['Frequency(Hz)'], errors='coerce')
            df.dropna(subset=['Frequency(Hz)'], inplace=True)
            
            # เก็บไฟล์ด้วย Key ที่เป็นตัวพิมพ์ใหญ่ ('101.CSV', '102.CSV', ...)
            dfs[it["name"].upper()] = df
        except Exception as e:
            print(f"❌ Failed to read {it['name']}: {e}")
            
    return dfs

# ==========================================
# 2. ดึงข้อมูล
# ==========================================
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN") 
owner = "NuchPunnawichP"
repo = "Senior_Project_CU"
path = "ThirdDataset/07"  # ใส่โฟลเดอร์ใน Github ที่มีไฟล์ข้อมูล 101-104 ครับ

# โหลดข้อมูลมาไว้ในตัวแปร dataframes
dataframes = load_csvs_from_github(owner, repo, path, ref="main", token=GITHUB_TOKEN)
print(f"\n✅ ดึงไฟล์สำเร็จ {len(dataframes)} ไฟล์: {list(dataframes.keys())}\n")

# ==========================================
# 3. สร้างโฟลเดอร์ชื่อ "10X" สำหรับเก็บรูปภาพ
# ==========================================
FOLDER_NAME = "10X"
os.makedirs(FOLDER_NAME, exist_ok=True)
print(f"📁 สร้าง/ตรวจสอบโฟลเดอร์สำหรับบันทึกรูป: '{FOLDER_NAME}' เรียบร้อยแล้ว\n")

# ==========================================
# 4. วาดกราฟและบันทึกลงในโฟลเดอร์ทีละความถี่
# ==========================================
target_files = ['101.CSV', '102.CSV', '103.CSV', '104.CSV']
file_labels = ['101', '102', '103', '104']

# แกน X ใช้ตัวเลข 1, 2, 3, 4 (มี 4 ไฟล์) เพื่อคำนวณ Linear Regression ได้
x = np.arange(1, 5) 

# ดึงรายการความถี่ทั้งหมดจากไฟล์แรก (101.CSV) มาใช้เป็นหลัก 
# จะได้เป๊ะกับจุดทศนิยมที่มีอยู่ในไฟล์ ไม่ต้องกะเอง
if '101.CSV' in dataframes:
    frequencies = dataframes['101.CSV']['Frequency(Hz)'].dropna().unique()
else:
    frequencies = []

# วนลูปวาดกราฟตามความถี่ทั้งหมดที่มี
for freq in frequencies:
    cp_vals = []
    
    # ดึงค่า Cp จากไฟล์ 401 ถึง 404 ที่ความถี่เดียวกัน
    for f_name in target_files:
        if f_name in dataframes:
            df = dataframes[f_name]
            # ใช้ np.isclose เผื่อค่าทศนิยมคลาดเคลื่อน
            row = df[np.isclose(df['Frequency(Hz)'], freq)]
            if not row.empty:
                # เปลี่ยนไปดึงค่า 'Cp(F)-data' แทน
                cp_vals.append(row['Cp(F)-data'].values[0])
            else:
                cp_vals.append(np.nan)
        else:
            cp_vals.append(np.nan)
            
    y = np.array(cp_vals)
    valid_idx = ~np.isnan(y) # เช็คว่าจุดไหนไม่มีค่า (NaN)
    
    # ถ้าไม่มีข้อมูลในย่านความถี่นี้เลย ให้ข้ามไป จะได้ไม่ Error
    if np.sum(valid_idx) == 0:
        continue
    
    # วาดกราฟ
    plt.figure(figsize=(10, 6))
    plt.plot(x, y, 'bo-', label='Cp(F)-data', markersize=8)
    
    # หาค่าเทรนด์ (Linear Regression)
    if np.sum(valid_idx) > 1:
        slope, intercept, r_value, _, _ = linregress(x[valid_idx], y[valid_idx])
        plt.plot(x, slope * x + intercept, 'r--', label=f'Linear fit (R²={r_value**2:.4f})')
        title_text = f'Capacitance (Cp) Trend across files at {freq:.2f} Hz\nEquation: y = {slope:.2e}x + {intercept:.2e}'
    else:
        title_text = f'Capacitance (Cp) Trend across files at {freq:.2f} Hz'
        
    # ใส่ตัวเลขค่า Cp กํากับบนจุดแต่ละจุด
    for i, val in enumerate(y):
        if not np.isnan(val):
            plt.annotate(f'{val:.2e}', 
                         (x[i], val),
                         textcoords="offset points", 
                         xytext=(0, 15), 
                         ha='center', fontsize=10)

    # จัดการ Label 
    plt.title(title_text, fontsize=14)
    plt.xticks(x, file_labels, fontsize=12) # เปลี่ยนเลขแกน X ให้เป็นชื่อไฟล์ '101' ถึง '104'
    plt.xlabel('File Number (101 to 104)', fontsize=12)
    plt.ylabel('Cp(F)-data', fontsize=12) # เปลี่ยนชื่อแกน Y
    
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.tight_layout()
    
    # กำหนดชื่อไฟล์และนำไปต่อกับ Path โฟลเดอร์ (ใช้ชื่อไฟล์เป็นค่า int ของ freq)
    filename = os.path.join(FOLDER_NAME, f'trend_at_{int(freq)}Hz.png')
    
    # สั่งบันทึกไฟล์กราฟลงในโฟลเดอร์
    plt.savefig(filename)
    # print(f"✅ บันทึกกราฟ {filename}") # ขอซ่อนไว้เนื่องจากมันจะพิมพ์ออกมาทั้งหมด 100 บรรทัด
    
    # เคลียร์ Memory ออก ไม่ให้คอมค้าง
    plt.close()

print(f"\n🎉 บันทึกกราฟครบทั้งหมดเรียบร้อยแล้ว! (มีรูปทั้งหมด {len(frequencies)} รูปในโฟลเดอร์ '{FOLDER_NAME}')")

กำลังเชื่อมต่อ GitHub: https://api.github.com/repos/NuchPunnawichP/Senior_Project_CU/contents/ThirdDataset/07?ref=main
กำลังดาวน์โหลดและอ่านไฟล์: 101.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 102.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 103.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 104.CSV ...

✅ ดึงไฟล์สำเร็จ 4 ไฟล์: ['101.CSV', '102.CSV', '103.CSV', '104.CSV']

📁 สร้าง/ตรวจสอบโฟลเดอร์สำหรับบันทึกรูป: '10X' เรียบร้อยแล้ว


🎉 บันทึกกราฟครบทั้งหมดเรียบร้อยแล้ว! (มีรูปทั้งหมด 100 รูปในโฟลเดอร์ '10X')


In [12]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# ==========================================
# 1. ฟังก์ชันโหลดไฟล์จาก GitHub
# ==========================================
def load_csvs_from_github(owner, repo, path, ref="main", token=None):
    session = requests.Session()
    headers = {}
    if token:
        headers["Authorization"] = f"token {token}"
        
    api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}?ref={ref}"
    print(f"กำลังเชื่อมต่อ GitHub: {api_url}")
    
    resp = session.get(api_url, headers=headers)
    resp.raise_for_status()
    items = resp.json()
    
    # สนใจเฉพาะไฟล์ 201-204
    target_files = ['201.csv', '202.csv', '203.csv', '204.csv']
    csv_items = [it for it in items if it.get("type") == "file" and it.get("name", "").lower() in target_files]
    
    dfs = {}
    for it in csv_items:
        url = it.get("download_url") or f"https://raw.githubusercontent.com/{owner}/{repo}/{ref}/{path}/{it['name']}"
        print(f"กำลังดาวน์โหลดและอ่านไฟล์: {it['name']} ...")
        try:
            # ใช้ skiprows=4 เพื่อข้าม Header บรรทัดบน
            df = pd.read_csv(url, skiprows=4)
            df.columns = df.columns.str.strip() # ตัดช่องว่างด้านหน้า/หลังในชื่อคอลัมน์ออก
            
            # ลบแถวสุดท้ายที่มีคำว่า 'END' ทิ้ง
            df = df[df['Frequency(Hz)'] != 'END']
            
            # บังคับแปลงเป็นตัวเลข
            df['Frequency(Hz)'] = pd.to_numeric(df['Frequency(Hz)'], errors='coerce')
            df.dropna(subset=['Frequency(Hz)'], inplace=True)
            
            # เก็บไฟล์ด้วย Key ที่เป็นตัวพิมพ์ใหญ่ ('101.CSV', '102.CSV', ...)
            dfs[it["name"].upper()] = df
        except Exception as e:
            print(f"❌ Failed to read {it['name']}: {e}")
            
    return dfs

# ==========================================
# 2. ดึงข้อมูล
# ==========================================
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN") 
owner = "NuchPunnawichP"
repo = "Senior_Project_CU"
path = "ThirdDataset/07"  # ใส่โฟลเดอร์ใน Github ที่มีไฟล์ข้อมูล 101-104 ครับ

# โหลดข้อมูลมาไว้ในตัวแปร dataframes
dataframes = load_csvs_from_github(owner, repo, path, ref="main", token=GITHUB_TOKEN)
print(f"\n✅ ดึงไฟล์สำเร็จ {len(dataframes)} ไฟล์: {list(dataframes.keys())}\n")

# ==========================================
# 3. สร้างโฟลเดอร์ชื่อ "20X" สำหรับเก็บรูปภาพ
# ==========================================
FOLDER_NAME = "20X"
os.makedirs(FOLDER_NAME, exist_ok=True)
print(f"📁 สร้าง/ตรวจสอบโฟลเดอร์สำหรับบันทึกรูป: '{FOLDER_NAME}' เรียบร้อยแล้ว\n")

# ==========================================
# 4. วาดกราฟและบันทึกลงในโฟลเดอร์ทีละความถี่
# ==========================================
target_files = ['201.CSV', '202.CSV', '203.CSV', '204.CSV']
file_labels = ['201', '202', '203', '204']

# แกน X ใช้ตัวเลข 1, 2, 3, 4 (มี 4 ไฟล์) เพื่อคำนวณ Linear Regression ได้
x = np.arange(1, 5) 

# ดึงรายการความถี่ทั้งหมดจากไฟล์แรก (201.CSV) มาใช้เป็นหลัก 
# จะได้เป๊ะกับจุดทศนิยมที่มีอยู่ในไฟล์ ไม่ต้องกะเอง
if '201.CSV' in dataframes:
    frequencies = dataframes['201.CSV']['Frequency(Hz)'].dropna().unique()
else:
    frequencies = []

# วนลูปวาดกราฟตามความถี่ทั้งหมดที่มี
for freq in frequencies:
    cp_vals = []
    
    # ดึงค่า Cp จากไฟล์ 401 ถึง 404 ที่ความถี่เดียวกัน
    for f_name in target_files:
        if f_name in dataframes:
            df = dataframes[f_name]
            # ใช้ np.isclose เผื่อค่าทศนิยมคลาดเคลื่อน
            row = df[np.isclose(df['Frequency(Hz)'], freq)]
            if not row.empty:
                # เปลี่ยนไปดึงค่า 'Cp(F)-data' แทน
                cp_vals.append(row['Cp(F)-data'].values[0])
            else:
                cp_vals.append(np.nan)
        else:
            cp_vals.append(np.nan)
            
    y = np.array(cp_vals)
    valid_idx = ~np.isnan(y) # เช็คว่าจุดไหนไม่มีค่า (NaN)
    
    # ถ้าไม่มีข้อมูลในย่านความถี่นี้เลย ให้ข้ามไป จะได้ไม่ Error
    if np.sum(valid_idx) == 0:
        continue
    
    # วาดกราฟ
    plt.figure(figsize=(10, 6))
    plt.plot(x, y, 'bo-', label='Cp(F)-data', markersize=8)
    
    # หาค่าเทรนด์ (Linear Regression)
    if np.sum(valid_idx) > 1:
        slope, intercept, r_value, _, _ = linregress(x[valid_idx], y[valid_idx])
        plt.plot(x, slope * x + intercept, 'r--', label=f'Linear fit (R²={r_value**2:.4f})')
        title_text = f'Capacitance (Cp) Trend across files at {freq:.2f} Hz\nEquation: y = {slope:.2e}x + {intercept:.2e}'
    else:
        title_text = f'Capacitance (Cp) Trend across files at {freq:.2f} Hz'
        
    # ใส่ตัวเลขค่า Cp กํากับบนจุดแต่ละจุด
    for i, val in enumerate(y):
        if not np.isnan(val):
            plt.annotate(f'{val:.2e}', 
                         (x[i], val),
                         textcoords="offset points", 
                         xytext=(0, 15), 
                         ha='center', fontsize=10)

    # จัดการ Label 
    plt.title(title_text, fontsize=14)
    plt.xticks(x, file_labels, fontsize=12) # เปลี่ยนเลขแกน X ให้เป็นชื่อไฟล์ '101' ถึง '104'
    plt.xlabel('File Number (101 to 104)', fontsize=12)
    plt.ylabel('Cp(F)-data', fontsize=12) # เปลี่ยนชื่อแกน Y
    
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.tight_layout()
    
    # กำหนดชื่อไฟล์และนำไปต่อกับ Path โฟลเดอร์ (ใช้ชื่อไฟล์เป็นค่า int ของ freq)
    filename = os.path.join(FOLDER_NAME, f'trend_at_{int(freq)}Hz.png')
    
    # สั่งบันทึกไฟล์กราฟลงในโฟลเดอร์
    plt.savefig(filename)
    # print(f"✅ บันทึกกราฟ {filename}") # ขอซ่อนไว้เนื่องจากมันจะพิมพ์ออกมาทั้งหมด 100 บรรทัด
    
    # เคลียร์ Memory ออก ไม่ให้คอมค้าง
    plt.close()

print(f"\n🎉 บันทึกกราฟครบทั้งหมดเรียบร้อยแล้ว! (มีรูปทั้งหมด {len(frequencies)} รูปในโฟลเดอร์ '{FOLDER_NAME}')")

กำลังเชื่อมต่อ GitHub: https://api.github.com/repos/NuchPunnawichP/Senior_Project_CU/contents/ThirdDataset/07?ref=main
กำลังดาวน์โหลดและอ่านไฟล์: 201.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 202.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 203.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 204.CSV ...

✅ ดึงไฟล์สำเร็จ 4 ไฟล์: ['201.CSV', '202.CSV', '203.CSV', '204.CSV']

📁 สร้าง/ตรวจสอบโฟลเดอร์สำหรับบันทึกรูป: '20X' เรียบร้อยแล้ว


🎉 บันทึกกราฟครบทั้งหมดเรียบร้อยแล้ว! (มีรูปทั้งหมด 100 รูปในโฟลเดอร์ '20X')


In [13]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# ==========================================
# 1. ฟังก์ชันโหลดไฟล์จาก GitHub
# ==========================================
def load_csvs_from_github(owner, repo, path, ref="main", token=None):
    session = requests.Session()
    headers = {}
    if token:
        headers["Authorization"] = f"token {token}"
        
    api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}?ref={ref}"
    print(f"กำลังเชื่อมต่อ GitHub: {api_url}")
    
    resp = session.get(api_url, headers=headers)
    resp.raise_for_status()
    items = resp.json()
    
    # สนใจเฉพาะไฟล์ 301-304
    target_files = ['301.csv', '302.csv', '303.csv', '304.csv']
    csv_items = [it for it in items if it.get("type") == "file" and it.get("name", "").lower() in target_files]
    
    dfs = {}
    for it in csv_items:
        url = it.get("download_url") or f"https://raw.githubusercontent.com/{owner}/{repo}/{ref}/{path}/{it['name']}"
        print(f"กำลังดาวน์โหลดและอ่านไฟล์: {it['name']} ...")
        try:
            # ใช้ skiprows=4 เพื่อข้าม Header บรรทัดบน
            df = pd.read_csv(url, skiprows=4)
            df.columns = df.columns.str.strip() # ตัดช่องว่างด้านหน้า/หลังในชื่อคอลัมน์ออก
            
            # ลบแถวสุดท้ายที่มีคำว่า 'END' ทิ้ง
            df = df[df['Frequency(Hz)'] != 'END']
            
            # บังคับแปลงเป็นตัวเลข
            df['Frequency(Hz)'] = pd.to_numeric(df['Frequency(Hz)'], errors='coerce')
            df.dropna(subset=['Frequency(Hz)'], inplace=True)
            
            # เก็บไฟล์ด้วย Key ที่เป็นตัวพิมพ์ใหญ่ ('101.CSV', '102.CSV', ...)
            dfs[it["name"].upper()] = df
        except Exception as e:
            print(f"❌ Failed to read {it['name']}: {e}")
            
    return dfs

# ==========================================
# 2. ดึงข้อมูล
# ==========================================
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN") 
owner = "NuchPunnawichP"
repo = "Senior_Project_CU"
path = "ThirdDataset/07"  # ใส่โฟลเดอร์ใน Github ที่มีไฟล์ข้อมูล 101-104 ครับ

# โหลดข้อมูลมาไว้ในตัวแปร dataframes
dataframes = load_csvs_from_github(owner, repo, path, ref="main", token=GITHUB_TOKEN)
print(f"\n✅ ดึงไฟล์สำเร็จ {len(dataframes)} ไฟล์: {list(dataframes.keys())}\n")

# ==========================================
# 3. สร้างโฟลเดอร์ชื่อ "30X" สำหรับเก็บรูปภาพ
# ==========================================
FOLDER_NAME = "30X"
os.makedirs(FOLDER_NAME, exist_ok=True)
print(f"📁 สร้าง/ตรวจสอบโฟลเดอร์สำหรับบันทึกรูป: '{FOLDER_NAME}' เรียบร้อยแล้ว\n")

# ==========================================
# 4. วาดกราฟและบันทึกลงในโฟลเดอร์ทีละความถี่
# ==========================================
target_files = ['301.CSV', '302.CSV', '303.CSV', '304.CSV']
file_labels = ['301', '302', '303', '304']

# แกน X ใช้ตัวเลข 1, 2, 3, 4 (มี 4 ไฟล์) เพื่อคำนวณ Linear Regression ได้
x = np.arange(1, 5) 

# ดึงรายการความถี่ทั้งหมดจากไฟล์แรก (301.CSV) มาใช้เป็นหลัก 
# จะได้เป๊ะกับจุดทศนิยมที่มีอยู่ในไฟล์ ไม่ต้องกะเอง
if '301.CSV' in dataframes:
    frequencies = dataframes['301.CSV']['Frequency(Hz)'].dropna().unique()
else:
    frequencies = []

# วนลูปวาดกราฟตามความถี่ทั้งหมดที่มี
for freq in frequencies:
    cp_vals = []
    
    # ดึงค่า Cp จากไฟล์ 401 ถึง 404 ที่ความถี่เดียวกัน
    for f_name in target_files:
        if f_name in dataframes:
            df = dataframes[f_name]
            # ใช้ np.isclose เผื่อค่าทศนิยมคลาดเคลื่อน
            row = df[np.isclose(df['Frequency(Hz)'], freq)]
            if not row.empty:
                # เปลี่ยนไปดึงค่า 'Cp(F)-data' แทน
                cp_vals.append(row['Cp(F)-data'].values[0])
            else:
                cp_vals.append(np.nan)
        else:
            cp_vals.append(np.nan)
            
    y = np.array(cp_vals)
    valid_idx = ~np.isnan(y) # เช็คว่าจุดไหนไม่มีค่า (NaN)
    
    # ถ้าไม่มีข้อมูลในย่านความถี่นี้เลย ให้ข้ามไป จะได้ไม่ Error
    if np.sum(valid_idx) == 0:
        continue
    
    # วาดกราฟ
    plt.figure(figsize=(10, 6))
    plt.plot(x, y, 'bo-', label='Cp(F)-data', markersize=8)
    
    # หาค่าเทรนด์ (Linear Regression)
    if np.sum(valid_idx) > 1:
        slope, intercept, r_value, _, _ = linregress(x[valid_idx], y[valid_idx])
        plt.plot(x, slope * x + intercept, 'r--', label=f'Linear fit (R²={r_value**2:.4f})')
        title_text = f'Capacitance (Cp) Trend across files at {freq:.2f} Hz\nEquation: y = {slope:.2e}x + {intercept:.2e}'
    else:
        title_text = f'Capacitance (Cp) Trend across files at {freq:.2f} Hz'
        
    # ใส่ตัวเลขค่า Cp กํากับบนจุดแต่ละจุด
    for i, val in enumerate(y):
        if not np.isnan(val):
            plt.annotate(f'{val:.2e}', 
                         (x[i], val),
                         textcoords="offset points", 
                         xytext=(0, 15), 
                         ha='center', fontsize=10)

    # จัดการ Label 
    plt.title(title_text, fontsize=14)
    plt.xticks(x, file_labels, fontsize=12) # เปลี่ยนเลขแกน X ให้เป็นชื่อไฟล์ '101' ถึง '104'
    plt.xlabel('File Number (101 to 104)', fontsize=12)
    plt.ylabel('Cp(F)-data', fontsize=12) # เปลี่ยนชื่อแกน Y
    
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.tight_layout()
    
    # กำหนดชื่อไฟล์และนำไปต่อกับ Path โฟลเดอร์ (ใช้ชื่อไฟล์เป็นค่า int ของ freq)
    filename = os.path.join(FOLDER_NAME, f'trend_at_{int(freq)}Hz.png')
    
    # สั่งบันทึกไฟล์กราฟลงในโฟลเดอร์
    plt.savefig(filename)
    # print(f"✅ บันทึกกราฟ {filename}") # ขอซ่อนไว้เนื่องจากมันจะพิมพ์ออกมาทั้งหมด 100 บรรทัด
    
    # เคลียร์ Memory ออก ไม่ให้คอมค้าง
    plt.close()

print(f"\n🎉 บันทึกกราฟครบทั้งหมดเรียบร้อยแล้ว! (มีรูปทั้งหมด {len(frequencies)} รูปในโฟลเดอร์ '{FOLDER_NAME}')")

กำลังเชื่อมต่อ GitHub: https://api.github.com/repos/NuchPunnawichP/Senior_Project_CU/contents/ThirdDataset/07?ref=main
กำลังดาวน์โหลดและอ่านไฟล์: 301.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 302.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 303.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 304.CSV ...

✅ ดึงไฟล์สำเร็จ 4 ไฟล์: ['301.CSV', '302.CSV', '303.CSV', '304.CSV']

📁 สร้าง/ตรวจสอบโฟลเดอร์สำหรับบันทึกรูป: '30X' เรียบร้อยแล้ว


🎉 บันทึกกราฟครบทั้งหมดเรียบร้อยแล้ว! (มีรูปทั้งหมด 100 รูปในโฟลเดอร์ '30X')


In [14]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# ==========================================
# 1. ฟังก์ชันโหลดไฟล์จาก GitHub
# ==========================================
def load_csvs_from_github(owner, repo, path, ref="main", token=None):
    session = requests.Session()
    headers = {}
    if token:
        headers["Authorization"] = f"token {token}"
        
    api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}?ref={ref}"
    print(f"กำลังเชื่อมต่อ GitHub: {api_url}")
    
    resp = session.get(api_url, headers=headers)
    resp.raise_for_status()
    items = resp.json()
    
    # สนใจเฉพาะไฟล์ 401-404
    target_files = ['401.csv', '402.csv', '403.csv', '404.csv']
    csv_items = [it for it in items if it.get("type") == "file" and it.get("name", "").lower() in target_files]
    
    dfs = {}
    for it in csv_items:
        url = it.get("download_url") or f"https://raw.githubusercontent.com/{owner}/{repo}/{ref}/{path}/{it['name']}"
        print(f"กำลังดาวน์โหลดและอ่านไฟล์: {it['name']} ...")
        try:
            # ใช้ skiprows=4 เพื่อข้าม Header บรรทัดบน
            df = pd.read_csv(url, skiprows=4)
            df.columns = df.columns.str.strip() # ตัดช่องว่างด้านหน้า/หลังในชื่อคอลัมน์ออก
            
            # ลบแถวสุดท้ายที่มีคำว่า 'END' ทิ้ง
            df = df[df['Frequency(Hz)'] != 'END']
            
            # บังคับแปลงเป็นตัวเลข
            df['Frequency(Hz)'] = pd.to_numeric(df['Frequency(Hz)'], errors='coerce')
            df.dropna(subset=['Frequency(Hz)'], inplace=True)
            
            # เก็บไฟล์ด้วย Key ที่เป็นตัวพิมพ์ใหญ่ ('101.CSV', '102.CSV', ...)
            dfs[it["name"].upper()] = df
        except Exception as e:
            print(f"❌ Failed to read {it['name']}: {e}")
            
    return dfs

# ==========================================
# 2. ดึงข้อมูล
# ==========================================
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN") 
owner = "NuchPunnawichP"
repo = "Senior_Project_CU"
path = "ThirdDataset/07"  # ใส่โฟลเดอร์ใน Github ที่มีไฟล์ข้อมูล 101-104 ครับ

# โหลดข้อมูลมาไว้ในตัวแปร dataframes
dataframes = load_csvs_from_github(owner, repo, path, ref="main", token=GITHUB_TOKEN)
print(f"\n✅ ดึงไฟล์สำเร็จ {len(dataframes)} ไฟล์: {list(dataframes.keys())}\n")

# ==========================================
# 3. สร้างโฟลเดอร์ชื่อ "40X" สำหรับเก็บรูปภาพ
# ==========================================
FOLDER_NAME = "40X"
os.makedirs(FOLDER_NAME, exist_ok=True)
print(f"📁 สร้าง/ตรวจสอบโฟลเดอร์สำหรับบันทึกรูป: '{FOLDER_NAME}' เรียบร้อยแล้ว\n")

# ==========================================
# 4. วาดกราฟและบันทึกลงในโฟลเดอร์ทีละความถี่
# ==========================================
target_files = ['401.CSV', '402.CSV', '403.CSV', '404.CSV']
file_labels = ['401', '402', '403', '404']

# แกน X ใช้ตัวเลข 1, 2, 3, 4 (มี 4 ไฟล์) เพื่อคำนวณ Linear Regression ได้
x = np.arange(1, 5) 

# ดึงรายการความถี่ทั้งหมดจากไฟล์แรก (401.CSV) มาใช้เป็นหลัก 
# จะได้เป๊ะกับจุดทศนิยมที่มีอยู่ในไฟล์ ไม่ต้องกะเอง
if '401.CSV' in dataframes:
    frequencies = dataframes['401.CSV']['Frequency(Hz)'].dropna().unique()
else:
    frequencies = []

# วนลูปวาดกราฟตามความถี่ทั้งหมดที่มี
for freq in frequencies:
    cp_vals = []
    
    # ดึงค่า Cp จากไฟล์ 401 ถึง 404 ที่ความถี่เดียวกัน
    for f_name in target_files:
        if f_name in dataframes:
            df = dataframes[f_name]
            # ใช้ np.isclose เผื่อค่าทศนิยมคลาดเคลื่อน
            row = df[np.isclose(df['Frequency(Hz)'], freq)]
            if not row.empty:
                # เปลี่ยนไปดึงค่า 'Cp(F)-data' แทน
                cp_vals.append(row['Cp(F)-data'].values[0])
            else:
                cp_vals.append(np.nan)
        else:
            cp_vals.append(np.nan)
            
    y = np.array(cp_vals)
    valid_idx = ~np.isnan(y) # เช็คว่าจุดไหนไม่มีค่า (NaN)
    
    # ถ้าไม่มีข้อมูลในย่านความถี่นี้เลย ให้ข้ามไป จะได้ไม่ Error
    if np.sum(valid_idx) == 0:
        continue
    
    # วาดกราฟ
    plt.figure(figsize=(10, 6))
    plt.plot(x, y, 'bo-', label='Cp(F)-data', markersize=8)
    
    # หาค่าเทรนด์ (Linear Regression)
    if np.sum(valid_idx) > 1:
        slope, intercept, r_value, _, _ = linregress(x[valid_idx], y[valid_idx])
        plt.plot(x, slope * x + intercept, 'r--', label=f'Linear fit (R²={r_value**2:.4f})')
        title_text = f'Capacitance (Cp) Trend across files at {freq:.2f} Hz\nEquation: y = {slope:.2e}x + {intercept:.2e}'
    else:
        title_text = f'Capacitance (Cp) Trend across files at {freq:.2f} Hz'
        
    # ใส่ตัวเลขค่า Cp กํากับบนจุดแต่ละจุด
    for i, val in enumerate(y):
        if not np.isnan(val):
            plt.annotate(f'{val:.2e}', 
                         (x[i], val),
                         textcoords="offset points", 
                         xytext=(0, 15), 
                         ha='center', fontsize=10)

    # จัดการ Label 
    plt.title(title_text, fontsize=14)
    plt.xticks(x, file_labels, fontsize=12) # เปลี่ยนเลขแกน X ให้เป็นชื่อไฟล์ '101' ถึง '104'
    plt.xlabel('File Number (101 to 104)', fontsize=12)
    plt.ylabel('Cp(F)-data', fontsize=12) # เปลี่ยนชื่อแกน Y
    
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.tight_layout()
    
    # กำหนดชื่อไฟล์และนำไปต่อกับ Path โฟลเดอร์ (ใช้ชื่อไฟล์เป็นค่า int ของ freq)
    filename = os.path.join(FOLDER_NAME, f'trend_at_{int(freq)}Hz.png')
    
    # สั่งบันทึกไฟล์กราฟลงในโฟลเดอร์
    plt.savefig(filename)
    # print(f"✅ บันทึกกราฟ {filename}") # ขอซ่อนไว้เนื่องจากมันจะพิมพ์ออกมาทั้งหมด 100 บรรทัด
    
    # เคลียร์ Memory ออก ไม่ให้คอมค้าง
    plt.close()

print(f"\n🎉 บันทึกกราฟครบทั้งหมดเรียบร้อยแล้ว! (มีรูปทั้งหมด {len(frequencies)} รูปในโฟลเดอร์ '{FOLDER_NAME}')")

กำลังเชื่อมต่อ GitHub: https://api.github.com/repos/NuchPunnawichP/Senior_Project_CU/contents/ThirdDataset/07?ref=main
กำลังดาวน์โหลดและอ่านไฟล์: 401.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 402.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 403.CSV ...
กำลังดาวน์โหลดและอ่านไฟล์: 404.CSV ...

✅ ดึงไฟล์สำเร็จ 4 ไฟล์: ['401.CSV', '402.CSV', '403.CSV', '404.CSV']

📁 สร้าง/ตรวจสอบโฟลเดอร์สำหรับบันทึกรูป: '40X' เรียบร้อยแล้ว


🎉 บันทึกกราฟครบทั้งหมดเรียบร้อยแล้ว! (มีรูปทั้งหมด 100 รูปในโฟลเดอร์ '40X')
